# Transformers para la clasificación de emociones

Curso: Minería de Datos

Proyecto: Clasificación automática de emociones en publicaciones de jóvenes peruanos

### 1. Preparación del entorno

#### Verificar la GPU

Utilizaremos Google Colab para aprovechar su GPU y que el proceso sea mucho más rápido.

In [1]:
import torch
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo detectado: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[!] NO se detectó GPU. El entrenamiento será muy lento.")

Dispositivo detectado: cuda
GPU: Tesla T4
Memoria: 15.6 GB


#### Entorno de ejecución

In [2]:
import os
import sys

if os.path.exists('/content'):
    # Entorno de la nube
    print("Entorno de la nube (Colab) detectado.")
    print("Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    BASE_DIR = '/content'
    # Ruta a la carpeta donde se encuentra los datasets limpios
    DATOS_DIR = '/content/drive/MyDrive/Proyecto-Mineria-G4/prueba'
else:
    # Entorno local
    print("Entorno local detectado.")
    try:
        current_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        current_dir = os.getcwd()

    BASE_DIR = current_dir
    while BASE_DIR != os.path.dirname(BASE_DIR):
        if 'modelado' in os.listdir(BASE_DIR) and os.path.isdir(os.path.join(BASE_DIR, 'modelado')):
            break
        BASE_DIR = os.path.dirname(BASE_DIR)
        
    DATOS_DIR = os.path.join(BASE_DIR, 'modelado', 'datos')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

RESULTADOS_DIR = os.path.join(BASE_DIR, 'resultados_transformers')
os.makedirs(RESULTADOS_DIR, exist_ok=True)

print(f"Directorio base detectado: {BASE_DIR}")
print(f"Directorio de datos: {DATOS_DIR}")


Entorno de la nube (Colab) detectado.
Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio base detectado: /content
Directorio de datos: /content/drive/MyDrive/Proyecto-Mineria-G4/prueba


#### Instalación de dependencias

In [3]:
!pip install transformers datasets pysentimiento evaluate accelerate scikit-learn pandas matplotlib seaborn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 42.6 MB/s eta 0:00:00


#### Importación de librerías

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.preprocessing import LabelBinarizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    EarlyStoppingCallback
)
from accelerate import Accelerator

print("Todas las librerías se importaron correctamente.")

Todas las librerías se importaron correctamente.


### 2. Carga y estandarización de los datasets

Se cargan los 3 datasets preprocesados (YouTube parte 1, YouTube parte 2, TikTok) y se unifican en un solo dataframe.

In [5]:
import glob
import pandas as pd

archivos = [
    os.path.join(DATOS_DIR, 'dataset_procesado_tiktok_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte02.csv')
]

dfs = []
for f in archivos:
    if os.path.exists(f):
        print(f"Cargando {os.path.basename(f)}...")
        temp_df = pd.read_csv(f, sep=';', encoding='utf-8')
        temp_df.columns = temp_df.columns.str.strip().str.replace('"', '').str.lower()
        dfs.append(temp_df)

if not dfs:
    raise ValueError("No se encontraron los datasets en modelado/datos/")

df = pd.concat(dfs, ignore_index=True)
df['emocion'] = df['emocion'].astype(str).str.strip().str.replace('"', '').str.capitalize()
df['emocion'] = df['emocion'].replace({'Alegría': 'Alegria'})

# Filtrar 6 emociones maestras
emociones_validas = ['Sorpresa', 'Miedo', 'Alegria', 'Tristeza']
df = df[df['emocion'].isin(emociones_validas)]
# Usamos la nueva columna del dataset consolidado
df = df.dropna(subset=['content_transformers', 'emocion'])

print(f"\nDataset unificado: {len(df)} registros")
print(f"Distribución de emociones:\n{df['emocion'].value_counts()}")


Cargando dataset_procesado_tiktok_parte01.csv...
Cargando dataset_procesado_youtube_parte01.csv...
Cargando dataset_procesado_youtube_parte02.csv...

Dataset unificado: 9065 registros
Distribución de emociones:
emocion
Tristeza    2684
Alegria     2329
Sorpresa    2048
Miedo       2004
Name: count, dtype: int64


### 3. División y preparación del conjunto de datos

In [6]:
# Se utiliza la columna content_transformers
textos = df['content_transformers'].astype(str).tolist()
etiquetas = df['emocion'].tolist()
clases = sorted(df['emocion'].unique().tolist())
print(f"Clases: {clases}")
total = len(etiquetas)
print(f"Total de registros: {total}")
# División: 80% entrenamiento / 20% prueba
# 1er corte: 20% prueba final
train_texts_temp, test_texts, train_labels_temp, test_labels = train_test_split(
    textos, etiquetas, test_size=0.20, random_state=42, stratify=etiquetas
)
# 2do corte: 30% del 80% → validación (~24% del total)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_temp, train_labels_temp, test_size=0.30, random_state=42,
    stratify=train_labels_temp
)
print(f"\nEntrenamiento Efectivo: {len(train_texts)} registros (~{len(train_texts)/total*100:.1f}%)")
print(f"Validación:             {len(val_texts)} registros (~{len(val_texts)/total*100:.1f}%)")
print(f"Prueba y Evaluación:    {len(test_texts)} registros (~{len(test_texts)/total*100:.1f}%)")
# Mapa de etiquetas a índices
label2id = {e: i for i, e in enumerate(clases)}
id2label  = {i: e for i, e in enumerate(clases)}
print(f"\nMapa de etiquetas: {label2id}")
train_labels_ids = [label2id[l] for l in train_labels]
val_labels_ids   = [label2id[l] for l in val_labels]
test_labels_ids  = [label2id[l] for l in test_labels]

Clases: ['Alegria', 'Miedo', 'Sorpresa', 'Tristeza']
Total de registros: 9065

Entrenamiento Efectivo: 5076 registros (~56.0%)
Validación:             2176 registros (~24.0%)
Prueba y Evaluación:    1813 registros (~20.0%)

Mapa de etiquetas: {'Alegria': 0, 'Miedo': 1, 'Sorpresa': 2, 'Tristeza': 3}


#### Pesos por clase (para manejar desbalance)

In [7]:
from sklearn.utils.class_weight import compute_class_weight

classes_array = np.array(clases)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes_array,
    y=np.array(train_labels)
)
class_weights_dict = {clases[i]: class_weights[i] for i in range(len(clases))}
pesos_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Pesos por clase (balanced):")
for c, w in class_weights_dict.items():
    print(f"   {c}: {w:.4f}")

Pesos por clase (balanced):
   Alegria: 0.9732
   Miedo: 1.1310
   Sorpresa: 1.1064
   Tristeza: 0.8443


#### Clase Dataset de PyTorch

In [8]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

## 4. Funciones de evaluación

Se definen las funciones para calcular métricas y generar los gráficos (matriz de confusión y curva ROC multiclase).

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision_macro': precision_score(labels, predictions, average='macro', zero_division=0),
        'recall_macro': recall_score(labels, predictions, average='macro', zero_division=0),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0)
    }


def generar_matriz_confusion(y_true, y_pred, clases, titulo, filename, ruta):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(clases)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=clases, yticklabels=clases)
    plt.title(f'Matriz de Confusión - {titulo}')
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.xticks(rotation=45)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa


def generar_curva_roc(y_true, y_scores, clases, titulo, filename, ruta):
    lb = LabelBinarizer()
    y_true_bin = lb.fit_transform(y_true)

    plt.figure(figsize=(10, 8))

    fpr_dict = {}
    tpr_dict = {}
    roc_auc = {}

    for i, clase in enumerate(clases):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc[i] = auc(fpr_dict[i], tpr_dict[i])
        plt.plot(fpr_dict[i], tpr_dict[i], lw=2,
                 label=f'{clase} (AUC = {roc_auc[i]:.3f})')

    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(len(clases))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(clases)):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= len(clases)
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, 'k--', lw=3,
             label=f'Macro-promedio (AUC = {macro_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'gray', linestyle=':', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title(f'Curva ROC - {titulo}')
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa, roc_auc, macro_auc


def reporte_por_clase(y_true, y_pred, clases):
    report = classification_report(y_true, y_pred, labels=range(len(clases)),
                                   target_names=clases, zero_division=0, output_dict=True)
    return report


def guardar_reporte_markdown(filename, ruta, nombre_modelo, metricas, cm_path, roc_path,
                             roc_auc_dict, macro_auc, reporte_clases, y_true, y_pred, clases):
    ruta_completa = os.path.join(ruta, filename)
    cm_rel = cm_path.replace('\\', '/')
    roc_rel = roc_path.replace('\\', '/')

    with open(ruta_completa, 'w', encoding='utf-8') as f:
        f.write(f"# Reporte de Evaluación - {nombre_modelo}\n\n")
        f.write(f"Resultados de métricas de rendimiento para el modelo {nombre_modelo}.\n\n")

        f.write("## Métricas Globales\n\n")
        f.write(f"- **Accuracy:** {metricas['accuracy']:.4f}\n")
        f.write(f"- **Precision (macro):** {metricas['precision_macro']:.4f}\n")
        f.write(f"- **Recall (macro):** {metricas['recall_macro']:.4f}\n")
        f.write(f"- **F1-Score (macro):** {metricas['f1_macro']:.4f}\n\n")

        f.write("## AUC (Área Bajo la Curva ROC)\n\n")
        for i, clase in enumerate(clases):
            f.write(f"- **{clase}:** {roc_auc_dict[i]:.4f}\n")
        f.write(f"- **Macro-promedio:** {macro_auc:.4f}\n\n")

        f.write("## Reporte por Clase\n\n")
        f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
        f.write("|-------|-----------|--------|----------|---------|\n")
        for clase in clases:
            r = reporte_clases[clase]
            f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
        f.write("\n")

        f.write("## Matriz de Confusión\n\n")
        f.write(f"![Matriz de Confusión {nombre_modelo}]({cm_rel})\n\n")

        f.write("## Curva ROC\n\n")
        f.write(f"![Curva ROC {nombre_modelo}]({roc_rel})\n\n")

        f.write("## Classification Report (detallado)\n\n")
        f.write("```text\n")
        f.write(classification_report(y_true, y_pred, labels=range(len(clases)),
                                      target_names=clases, zero_division=0))
        f.write("```\n")

    print(f"Reporte guardado: {ruta_completa}")
    return ruta_completa

#### Hiperparametros compartidos

In [10]:
MAX_LENGTH   = 128
BATCH_SIZE   = 16
EPOCHS       = 5       # early stopping controla el límite real
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200

### 5. Entrenamiento: BETO (BERT para corpus en español)

Modelo: [`dccuchile/bert-base-spanish-wwm-cased`](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased)

In [11]:
MODELO_BETO = 'dccuchile/bert-base-spanish-wwm-cased'
LEARNING_RATE_BETO = 3e-5

print(f"Configuración del Modelo Base:")
print(f"   Modelo: {MODELO_BETO}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE_BETO}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Warmup steps: {WARMUP_STEPS}")

Configuración del Modelo Base:
   Modelo: dccuchile/bert-base-spanish-wwm-cased
   Max length: 128
   Batch size: 16
   Epochs: 5
   Learning rate: 3e-05
   Weight decay: 0.01
   Warmup steps: 200


In [12]:
print("Cargando tokenizer...")
tokenizer_beto = AutoTokenizer.from_pretrained(MODELO_BETO)

print("Tokenizando textos de entrenamiento efectivo...")
train_encodings_beto = tokenizer_beto(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

print("Tokenizando textos de prueba...")
test_encodings_beto = tokenizer_beto(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

print("Tokenizando textos de prueba y validacion...")
val_encodings_beto = tokenizer_beto(
    val_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

train_dataset_beto = EmotionDataset(train_encodings_beto, train_labels_ids)
test_dataset_beto = EmotionDataset(test_encodings_beto, test_labels_ids)
val_dataset_beto = EmotionDataset(val_encodings_beto, val_labels_ids)

print(f"Train dataset: {len(train_dataset_beto)} muestras")
print(f"Test dataset: {len(test_dataset_beto)} muestras")
print(f"Val dataset: {len(val_dataset_beto)} muestras")

Cargando tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Tokenizando textos de entrenamiento efectivo...
Tokenizando textos de prueba...
Tokenizando textos de prueba y validacion...
Train dataset: 5076 muestras
Test dataset: 1813 muestras
Val dataset: 2176 muestras


#### Función para entrenar los modelos

In [13]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=pesos_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

print("CustomTrainer implentada.")

CustomTrainer implentada.


#### Entrenamiento del modelo

In [14]:
print("Cargando modelo pre-entrenado...")
model_beto = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BETO,
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id,
).to(device)

training_args_beto = TrainingArguments(
    output_dir='./resultados_beto',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs_beto',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    learning_rate=LEARNING_RATE_BETO,
)

trainer_beto = CustomTrainer(
    model=model_beto,
    args=training_args_beto,
    train_dataset=train_dataset_beto,
    eval_dataset=val_dataset_beto,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Iniciando entrenamiento...")
trainer_beto.train()

Cargando modelo pre-entrenado...


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Iniciando entrenamiento...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1272, training_loss=0.5725986991288528, metrics={'train_runtime': 539.5842, 'train_samples_per_second': 47.036, 'train_steps_per_second': 2.947, 'total_flos': 1335575699767296.0, 'train_loss': 0.5725986991288528, 'epoch': 4.0})

#### Evaluación del modelo

In [15]:
print("Evaluando BETO en los 3 conjuntos...")

# Entrenamiento Efectivo
beto_train_preds = trainer_beto.predict(train_dataset_beto)
beto_train_ids   = np.argmax(beto_train_preds.predictions, axis=-1)
beto_train_metrics = {
    'accuracy':        accuracy_score(train_labels_ids, beto_train_ids),
    'precision_macro': precision_score(train_labels_ids, beto_train_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(train_labels_ids, beto_train_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(train_labels_ids, beto_train_ids, average='macro', zero_division=0),
}
print(f"[Train] Accuracy: {beto_train_metrics['accuracy']:.4f} | F1: {beto_train_metrics['f1_macro']:.4f}")

# Validación
beto_val_preds = trainer_beto.predict(val_dataset_beto)
beto_val_ids   = np.argmax(beto_val_preds.predictions, axis=-1)
beto_val_metrics = {
    'accuracy':        accuracy_score(val_labels_ids, beto_val_ids),
    'precision_macro': precision_score(val_labels_ids, beto_val_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(val_labels_ids, beto_val_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(val_labels_ids, beto_val_ids, average='macro', zero_division=0),
}
print(f"[Val]   Accuracy: {beto_val_metrics['accuracy']:.4f} | F1: {beto_val_metrics['f1_macro']:.4f}")

# Prueba y Evaluación 
beto_test_preds = trainer_beto.predict(test_dataset_beto)
beto_logits     = beto_test_preds.predictions
beto_preds_ids  = np.argmax(beto_logits, axis=-1)
beto_metrics = {
    'accuracy':        accuracy_score(test_labels_ids, beto_preds_ids),
    'precision_macro': precision_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
}
print(f"[Test]  Accuracy: {beto_metrics['accuracy']:.4f} | F1: {beto_metrics['f1_macro']:.4f}")

# Gráficas (sobre test)
beto_cm_path = generar_matriz_confusion(test_labels_ids, beto_preds_ids, clases, 'BETO', 'cm_beto.png', RESULTADOS_DIR)
beto_report  = reporte_por_clase(test_labels_ids, beto_preds_ids, clases)
beto_probas  = torch.nn.functional.softmax(torch.tensor(beto_logits), dim=-1).numpy()
beto_roc_path, beto_auc_dict, beto_macro_auc = generar_curva_roc(
    test_labels_ids, beto_probas, clases, 'BETO', 'roc_beto.png', RESULTADOS_DIR
)

Evaluando BETO en los 3 conjuntos...


[Train] Accuracy: 0.8930 | F1: 0.8929


[Val]   Accuracy: 0.7247 | F1: 0.7225


[Test]  Accuracy: 0.7253 | F1: 0.7246


### 6. Entrenamiento: RoBERTuito (RoBERTa en español)

Modelo: [`pysentimiento/robertuito-base-uncased`](https://huggingface.co/pysentimiento/robertuito-base-uncased)

In [16]:
from pysentimiento.preprocessing import preprocess_tweet

MODELO_ROBERTA = 'pysentimiento/robertuito-base-uncased'
LEARNING_RATE_ROBERTA = 1e-5

print(f"Configuración del Modelo Base:")
print(f"   Modelo: {MODELO_ROBERTA}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE_ROBERTA}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Warmup steps: {WARMUP_STEPS}")

Configuración del Modelo Base:
   Modelo: pysentimiento/robertuito-base-uncased
   Max length: 128
   Batch size: 16
   Epochs: 5
   Learning rate: 1e-05
   Weight decay: 0.01
   Warmup steps: 200


In [17]:
print("Cargando tokenizer RoBERTa...")
tokenizer_roberta = AutoTokenizer.from_pretrained('pysentimiento/robertuito-base-uncased')

train_texts_proc = [preprocess_tweet(t) for t in train_texts]
test_texts_proc  = [preprocess_tweet(t) for t in test_texts]

print("Tokenizando textos de entrenamiento efectivo...")
train_encodings_roberta = tokenizer_roberta(
    train_texts_proc,   # textos preprocesados
    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors=None
)

print("Tokenizando textos de validacion...")
test_encodings_roberta = tokenizer_roberta(
    test_texts_proc,    # textos preprocesados
    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors=None
)

print("Tokenizando textos de prueba y validacion...")
val_texts_proc = [preprocess_tweet(t) for t in val_texts]
val_encodings_roberta = tokenizer_roberta(
    val_texts_proc,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

train_dataset_roberta = EmotionDataset(train_encodings_roberta, train_labels_ids)
test_dataset_roberta = EmotionDataset(test_encodings_roberta, test_labels_ids)
val_dataset_roberta = EmotionDataset(val_encodings_roberta, val_labels_ids)

print(f"Train dataset: {len(train_dataset_roberta)} muestras")
print(f"Test dataset: {len(test_dataset_roberta)} muestras")
print(f"Val dataset: {len(val_dataset_roberta)} muestras")


Cargando tokenizer RoBERTa...


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/858k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Tokenizando textos de entrenamiento efectivo...
Tokenizando textos de validacion...
Tokenizando textos de prueba y validacion...
Train dataset: 5076 muestras
Test dataset: 1813 muestras
Val dataset: 2176 muestras


In [18]:
print("Cargando modelo RoBERTa...")
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODELO_ROBERTA,              
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)


training_args_roberta = TrainingArguments(
    output_dir='/content/checkpoints_roberta',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_ROBERTA,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_dir='/content/logs_roberta',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    save_total_limit=1,
)

trainer_roberta = CustomTrainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=train_dataset_roberta,
    eval_dataset=val_dataset_roberta,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Iniciando entrenamiento de RoBERTa...")
trainer_roberta.train()
print("Entrenamiento de RoBERTa completado.")



Cargando modelo RoBERTa...


model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Iniciando entrenamiento de RoBERTa...
{'loss': '1.395', 'grad_norm': '5.618', 'learning_rate': '2.45e-06', 'epoch': '0.1572'}
{'loss': '1.363', 'grad_norm': '4.893', 'learning_rate': '4.95e-06', 'epoch': '0.3145'}
{'loss': '1.307', 'grad_norm': '5.625', 'learning_rate': '7.45e-06', 'epoch': '0.4717'}
{'loss': '1.208', 'grad_norm': '6.202', 'learning_rate': '9.95e-06', 'epoch': '0.6289'}
{'loss': '1.102', 'grad_norm': '6.32', 'learning_rate': '9.647e-06', 'epoch': '0.7862'}
{'loss': '1.04', 'grad_norm': '6.975', 'learning_rate': '9.288e-06', 'epoch': '0.9434'}
{'eval_loss': '0.9206', 'eval_accuracy': '0.6622', 'eval_precision_macro': '0.6605', 'eval_recall_macro': '0.6618', 'eval_f1_macro': '0.661', 'eval_runtime': '4.021', 'eval_samples_per_second': '541.2', 'eval_steps_per_second': '33.83', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'loss': '0.6376', 'grad_norm': '11.3', 'learning_rate': '4.252e-06', 'epoch': '3.145'}
{'loss': '0.6446', 'grad_norm': '14.62', 'learning_rate': '3.892e-06', 'epoch': '3.302'}
{'loss': '0.587', 'grad_norm': '9.515', 'learning_rate': '3.532e-06', 'epoch': '3.459'}
{'loss': '0.6007', 'grad_norm': '8.213', 'learning_rate': '3.173e-06', 'epoch': '3.616'}
{'loss': '0.5872', 'grad_norm': '9.224', 'learning_rate': '2.813e-06', 'epoch': '3.774'}
{'loss': '0.6035', 'grad_norm': '7.665', 'learning_rate': '2.453e-06', 'epoch': '3.931'}
{'eval_loss': '0.7784', 'eval_accuracy': '0.7206', 'eval_precision_macro': '0.7191', 'eval_recall_macro': '0.7185', 'eval_f1_macro': '0.7186', 'eval_runtime': '4.022', 'eval_samples_per_second': '541', 'eval_steps_per_second': '33.81', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.6221', 'grad_norm': '11.95', 'learning_rate': '2.094e-06', 'epoch': '4.088'}
{'loss': '0.5517', 'grad_norm': '8.395', 'learning_rate': '1.734e-06', 'epoch': '4.245'}
{'loss': '0.5445', 'grad_norm': '10.7', 'learning_rate': '1.374e-06', 'epoch': '4.403'}
{'loss': '0.5611', 'grad_norm': '11.27', 'learning_rate': '1.014e-06', 'epoch': '4.56'}
{'loss': '0.5664', 'grad_norm': '8.892', 'learning_rate': '6.547e-07', 'epoch': '4.717'}
{'loss': '0.588', 'grad_norm': '9.63', 'learning_rate': '2.95e-07', 'epoch': '4.874'}
{'eval_loss': '0.7812', 'eval_accuracy': '0.7206', 'eval_precision_macro': '0.7194', 'eval_recall_macro': '0.7185', 'eval_f1_macro': '0.7186', 'eval_runtime': '3.894', 'eval_samples_per_second': '558.9', 'eval_steps_per_second': '34.93', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '241.2', 'train_samples_per_second': '105.2', 'train_steps_per_second': '6.592', 'train_loss': '0.7876', 'epoch': '5'}
Entrenamiento de RoBERTa completado.


In [19]:
print("Evaluando RoBERTa en los 3 conjuntos...")

# Entrenamiento Efectivo
roberta_train_preds = trainer_roberta.predict(train_dataset_roberta)
roberta_train_ids   = np.argmax(roberta_train_preds.predictions, axis=-1)
roberta_train_metrics = {
    'accuracy':        accuracy_score(train_labels_ids, roberta_train_ids),
    'precision_macro': precision_score(train_labels_ids, roberta_train_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(train_labels_ids, roberta_train_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(train_labels_ids, roberta_train_ids, average='macro', zero_division=0),
}
print(f"[Train] Accuracy: {roberta_train_metrics['accuracy']:.4f} | F1: {roberta_train_metrics['f1_macro']:.4f}")

# Validación
roberta_val_preds = trainer_roberta.predict(val_dataset_roberta)
roberta_val_ids   = np.argmax(roberta_val_preds.predictions, axis=-1)
roberta_val_metrics = {
    'accuracy':        accuracy_score(val_labels_ids, roberta_val_ids),
    'precision_macro': precision_score(val_labels_ids, roberta_val_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(val_labels_ids, roberta_val_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(val_labels_ids, roberta_val_ids, average='macro', zero_division=0),
}
print(f"[Val]   Accuracy: {roberta_val_metrics['accuracy']:.4f} | F1: {roberta_val_metrics['f1_macro']:.4f}")

# Prueba y Evaluación
roberta_test_preds = trainer_roberta.predict(test_dataset_roberta)
roberta_logits     = roberta_test_preds.predictions
roberta_preds_ids  = np.argmax(roberta_logits, axis=-1)
roberta_metrics = {
    'accuracy':        accuracy_score(test_labels_ids, roberta_preds_ids),
    'precision_macro': precision_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'recall_macro':    recall_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'f1_macro':        f1_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
}
print(f"[Test]  Accuracy: {roberta_metrics['accuracy']:.4f} | F1: {roberta_metrics['f1_macro']:.4f}")

# Gráficas (sobre test)
roberta_cm_path = generar_matriz_confusion(test_labels_ids, roberta_preds_ids, clases, 'RoBERTa', 'cm_roberta.png', RESULTADOS_DIR)
roberta_report  = reporte_por_clase(test_labels_ids, roberta_preds_ids, clases)
roberta_probas  = torch.nn.functional.softmax(torch.tensor(roberta_logits), dim=-1).numpy()
roberta_roc_path, roberta_auc_dict, roberta_macro_auc = generar_curva_roc(
    test_labels_ids, roberta_probas, clases, 'RoBERTa', 'roc_roberta.png', RESULTADOS_DIR
)

Evaluando RoBERTa en los 3 conjuntos...
[Train] Accuracy: 0.8262 | F1: 0.8258
[Val]   Accuracy: 0.7206 | F1: 0.7186
[Test]  Accuracy: 0.7248 | F1: 0.7235



### 7. Exportación del reporte

In [ ]:
cm_beto_rel = beto_cm_path.replace('\\', '/')
cm_roberta_rel = roberta_cm_path.replace('\\', '/')
roc_beto_rel = beto_roc_path.replace('\\', '/')
roc_roberta_rel = roberta_roc_path.replace('\\', '/')

ruta_reporte = os.path.join(RESULTADOS_DIR, 'reporte_transformers.md')

with open(ruta_reporte, 'w', encoding='utf-8') as f:
    f.write("# Reporte de evaluación de modelos Transformer\n\n")
    f.write("Resultados de métricas de rendimiento para los modelos BETO (BERT en Español) y RoBERTa (RoBERTuito).\n\n")

    f.write("---\n\n")
    f.write("## Comparativa Global\n\n")
    f.write("| Métrica | BETO | RoBERTa |\n")
    f.write("|---------|------|---------|\n")
    f.write(f"| Accuracy | {beto_metrics['accuracy']:.4f} | {roberta_metrics['accuracy']:.4f} |\n")
    f.write(f"| Precision (macro) | {beto_metrics['precision_macro']:.4f} | {roberta_metrics['precision_macro']:.4f} |\n")
    f.write(f"| Recall (macro) | {beto_metrics['recall_macro']:.4f} | {roberta_metrics['recall_macro']:.4f} |\n")
    f.write(f"| F1-Score (macro) | {beto_metrics['f1_macro']:.4f} | {roberta_metrics['f1_macro']:.4f} |\n")
    f.write(f"| AUC (macro) | {beto_macro_auc:.4f} | {roberta_macro_auc:.4f} |\n\n")

    f.write("---\n\n")
    f.write("## BETO\n\n")
    f.write("### Resultados en Entrenamiento Efectivo (~56%)\n\n")
    f.write(f"- **Accuracy:** {beto_train_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {beto_train_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {beto_train_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {beto_train_metrics['f1_macro']:.4f}\n\n")

    f.write("### Resultados en Validación (~24%)\n\n")
    f.write(f"- **Accuracy:** {beto_val_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {beto_val_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {beto_val_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {beto_val_metrics['f1_macro']:.4f}\n\n")

    f.write("### Resultados en Prueba y Evaluación (20%)\n\n")
    f.write(f"- **Accuracy:** {beto_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {beto_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {beto_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {beto_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {beto_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {beto_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = beto_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión BETO]({cm_beto_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC BETO]({roc_beto_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, beto_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

    f.write("---\n\n")
    f.write("## RoBERTa\n\n")
    f.write("### Resultados en Entrenamiento Efectivo (~56%)\n\n")
    f.write(f"- **Accuracy:** {roberta_train_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {roberta_train_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {roberta_train_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {roberta_train_metrics['f1_macro']:.4f}\n\n")

    f.write("### Resultados en Validación (~24%)\n\n")
    f.write(f"- **Accuracy:** {roberta_val_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {roberta_val_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {roberta_val_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {roberta_val_metrics['f1_macro']:.4f}\n\n")

    f.write("### Resultados en Prueba y Evaluación (20%)\n\n")
    f.write(f"- **Accuracy:** {roberta_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {roberta_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {roberta_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {roberta_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {roberta_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {roberta_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = roberta_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión RoBERTa]({cm_roberta_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC RoBERTa]({roc_roberta_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, roberta_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

print(f"\nReporte guardado: {ruta_reporte}")
print("\nPROCESO COMPLETADO")
print("Se generaron:")
print(f"  - {os.path.basename(beto_cm_path)}")
print(f"  - {os.path.basename(beto_roc_path)}")
print(f"  - {os.path.basename(roberta_cm_path)}")
print(f"  - {os.path.basename(roberta_roc_path)}")
print(f"  - reporte_transformers.md")



Reporte guardado: /content/resultados_transformers/reporte_transformers.md

PROCESO COMPLETADO
Se generaron:
  - cm_beto.png
  - roc_beto.png
  - cm_roberta.png
  - roc_roberta.png
  - reporte_transformers.md
